# Homework 4 — Evaluation

Generate a ground-truth dataset and use it to compare keyword, vector and hybrid
search with Hit Rate and MRR. Continues from homework 2 (same 72 pages at commit
`8c1834d`, same 295 chunks).

In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

from embedder import Embedder
from evaluation_utils import llm_structured

load_dotenv()
embedder = Embedder()

## Load the lessons and chunks

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)
len(documents), len(chunks)

(72, 295)

## Q1 — Generating questions

For each of the first 3 pages, ask the LLM for 5 questions and read the input
tokens. Average them.

In [3]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

client = OpenAI()
input_tokens = []
for doc in documents[:3]:
    _, usage = llm_structured(client, data_gen_instructions, json.dumps(doc), Questions)
    input_tokens.append(usage.input_tokens)

input_tokens, sum(input_tokens) / len(input_tokens)

([1020, 1286, 1753], 1353.0)

## Search over the chunks

Text (`Index`) and vector (`VectorSearch`), both keyed on `filename`.

In [4]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

X = embedder.encode_batch([c["content"] for c in chunks])
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    return vector_index.search(embedder.encode(query), num_results=num_results)


def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Ground truth

In [5]:
ground_truth = pd.read_csv("../../cohorts/2026/04-evaluation/ground-truth.csv").to_dict(orient="records")
len(ground_truth)

360

## Q2 — First result, text search

In [6]:
q0 = ground_truth[0]["question"]
text_search(q0)[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

## Q3 — First result, vector search

This question was generated from `01-intro.md`.

In [7]:
vector_search(q0)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

## Evaluation metrics

A result is a hit when its `filename` matches the question's `filename`.

In [8]:
def compute_relevance(q, search_function):
    correct = q["filename"]
    results = search_function(q["question"])
    return [int(d["filename"] == correct) for d in results]


def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)


def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance)


def evaluate(ground_truth, search_function):
    relevance = [compute_relevance(q, search_function) for q in ground_truth]
    return {"hit_rate": hit_rate(relevance), "mrr": mrr(relevance)}

## Q4 — Text search

In [9]:
evaluate(ground_truth, text_search)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5 — Vector search

In [10]:
evaluate(ground_truth, vector_search)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q6 — Tuning hybrid search

Best `k` by MRR (ties → smallest `k`).

In [11]:
{k: evaluate(ground_truth, lambda q, k=k: hybrid_search(q, k=k))["mrr"] for k in [1, 50, 100, 200]}

{1: 0.6481944444444449,
 50: 0.637916666666667,
 100: 0.637916666666667,
 200: 0.637916666666667}

## Answers

| Q | Answer |
|---|--------|
| Q1 | ~1400 input tokens |
| Q2 | `01-agentic-rag/lessons/03-rag.md` |
| Q3 | `01-agentic-rag/lessons/01-intro.md` |
| Q4 | 0.76 hit rate |
| Q5 | 0.55 MRR |
| Q6 | k = 1 |